# 🏀 Pacers Data Engineering & Analytics Notebook

This notebook demonstrates the complete setup and exploratory data analysis for the Indiana Pacers NBA analytics project. It covers:

1. **Environment Setup and Verification**
2. **Project Structure Creation** 
3. **Git Repository Initialization**
4. **Conda Environment Configuration**
5. **Data Directory Preparation**
6. **Configuration File Setup**
7. **Gitignore Implementation**
8. **Makefile Creation**
9. **Environment Testing**

---

## Project Overview

This is a production-grade data engineering repository for NBA analytics focused on the Indiana Pacers. The project processes:
- Player statistics and information
- Game schedules  
- Detailed boxscores
- Play-by-play data

**Tech Stack**: Python 3.10, pandas, pyarrow, fastparquet, numpy, sqlalchemy, tqdm, matplotlib, seaborn

## 1. Environment Setup and Verification

First, let's verify that our system meets the requirements for this data engineering project.

In [ ]:
import sys
import os
import subprocess
import platform
from pathlib import Path

# Check Python version
print(f"Python Version: {sys.version}")
print(f"Platform: {platform.system()} {platform.release()}")

# Check if conda is available
try:
    result = subprocess.run(['conda', '--version'], capture_output=True, text=True)
    print(f"Conda: {result.stdout.strip()}")
    conda_available = True
except FileNotFoundError:
    print(" Conda not found! Please install Anaconda or Miniconda.")
    conda_available = False

# Check current working directory and auto-navigate to project root if needed
current_dir = Path.cwd()
print(f"Current Directory: {current_dir}")

# Check if we're in the notebooks directory and need to go up one level
if current_dir.name == 'notebooks':
    project_root = current_dir.parent
    print(f" Detected we're in notebooks directory, checking parent: {project_root}")
else:
    project_root = current_dir

# Verify we're in the project root (or can find it)
project_indicators = ['src', 'data', 'environment.yml']
is_project_root = all((project_root / indicator).exists() for indicator in project_indicators)

if is_project_root:
    print(f" Project root found: {project_root}")
    # Change to project root directory for the rest of the notebook
    os.chdir(project_root)
    print(f" Changed working directory to: {Path.cwd()}")
else:
    print(f" Project root not found. Looking for: {project_indicators}")
    print("Please ensure you're running this notebook from the project directory or notebooks subdirectory.")

## 2. Project Structure Creation

Let's create and verify the complete folder structure for our Pacers analytics project.

In [ ]:
from pathlib import Path

# Define the project structure
project_structure = {
    'data': ['raw', 'interim', 'processed'],
    'db': [],
    'docs': [],
    'notebooks': [],
    'outputs': [],
    'scripts': [],
    'src': ['etl'],
    'sql': []
}

# Create directories and .gitkeep files
def create_project_structure():
    project_root = Path.cwd()
    
    for main_dir, subdirs in project_structure.items():
        # Create main directory
        main_path = project_root / main_dir
        main_path.mkdir(exist_ok=True)
        print(f"✅ Created: {main_path}")
        
        # Create subdirectories
        for subdir in subdirs:
            sub_path = main_path / subdir
            sub_path.mkdir(exist_ok=True)
            print(f"  ✅ Created: {sub_path}")
            
            # Create .gitkeep for data and db subdirectories
            if main_dir in ['data', 'db'] or (main_dir == 'outputs' and len(subdirs) == 0):
                gitkeep_path = sub_path / '.gitkeep' if subdirs else main_path / '.gitkeep'
                if not gitkeep_path.exists():
                    gitkeep_path.write_text(f"# This file ensures the {gitkeep_path.parent.name} directory is tracked by git\n")
                    print(f"    📄 Created: {gitkeep_path}")

# Execute structure creation
create_project_structure()

# Verify structure
print("\n📁 Current Project Structure:")
project_root = Path.cwd()
for item in sorted(project_root.iterdir()):
    if item.is_dir() and not item.name.startswith('.'):
        print(f"📁 {item.name}/")
        for subitem in sorted(item.iterdir()):
            if subitem.is_dir():
                print(f"  📁 {subitem.name}/")
            else:
                print(f"  📄 {subitem.name}")
        print()

## 3. Git Repository Initialization

Set up git repository configuration and initial commit structure.

In [ ]:
import subprocess

# Check if git is already initialized
git_dir = Path('.git')
if git_dir.exists():
    print("✅ Git repository already initialized")
else:
    # Initialize git repository
    try:
        subprocess.run(['git', 'init'], check=True, capture_output=True)
        print("✅ Git repository initialized")
    except subprocess.CalledProcessError as e:
        print(f"❌ Error initializing git: {e}")

# Check git status
try:
    result = subprocess.run(['git', 'status', '--porcelain'], capture_output=True, text=True)
    if result.returncode == 0:
        untracked_files = result.stdout.strip().split('\n') if result.stdout.strip() else []
        print(f"📄 Untracked files: {len(untracked_files)}")
        
        # Show first few untracked files
        for file in untracked_files[:5]:
            print(f"  - {file}")
        if len(untracked_files) > 5:
            print(f"  ... and {len(untracked_files) - 5} more")
    
    # Check current branch
    result = subprocess.run(['git', 'branch', '--show-current'], capture_output=True, text=True)
    if result.returncode == 0 and result.stdout.strip():
        print(f"📋 Current branch: {result.stdout.strip()}")
    else:
        print("📋 No commits yet")
        
except subprocess.CalledProcessError:
    print("❌ Git not available or not in a git repository")

# Display git configuration recommendations
print("\n🔧 Recommended Git Configuration:")
print("git config --global user.name 'Your Name'")
print("git config --global user.email 'your.email@example.com'")
print("git remote add origin https://github.com/yourusername/Pacers-Pipeline-And-Dashboard.git")

## 4. Conda Environment Configuration

Create and configure the `pacers_de` conda environment with all required dependencies.

In [ ]:
# Check if environment.yml exists
env_file = Path('environment.yml')
if env_file.exists():
    print("✅ environment.yml found")
    print(f"📄 Content preview:")
    with open(env_file, 'r') as f:
        content = f.read()
        print(content[:500] + "..." if len(content) > 500 else content)
else:
    print("❌ environment.yml not found - should be created in project setup")

# Check if pacers_de environment exists
try:
    result = subprocess.run(['conda', 'env', 'list'], capture_output=True, text=True)
    if 'pacers_de' in result.stdout:
        print("✅ pacers_de environment already exists")
        
        # Get environment details
        result = subprocess.run(['conda', 'list', '-n', 'pacers_de'], capture_output=True, text=True)
        if result.returncode == 0:
            packages = [line.split()[0] for line in result.stdout.split('\n') 
                       if line and not line.startswith('#') and len(line.split()) > 0]
            required_packages = ['pandas', 'numpy', 'pyarrow', 'fastparquet', 'sqlalchemy', 'tqdm', 'matplotlib', 'seaborn']
            
            print(f"📦 Total packages installed: {len(packages)}")
            print("🔍 Required packages status:")
            for pkg in required_packages:
                status = "✅" if any(pkg in p for p in packages) else "❌"
                print(f"  {status} {pkg}")
    else:
        print("❌ pacers_de environment not found")
        print("💡 To create: conda env create -f environment.yml")
        
except subprocess.CalledProcessError:
    print("❌ Error checking conda environments")

## 5. Data Directory Preparation

Set up data directories and verify structure for NBA parquet files.

In [ ]:
# Check data directory structure and files
data_dir = Path('data')
required_files = [
    'data/raw/2024_NBA_Players.parquet',
    'data/raw/2024_NBA_Schedule.parquet',
    'data/raw/Boxscores/',
    'data/raw/PBP/'
]

print("📊 Data Directory Status:")
print("=" * 40)

for file_path in required_files:
    path = Path(file_path)
    if path.exists():
        if path.is_file():
            size_mb = path.stat().st_size / (1024 * 1024)
            print(f"✅ {file_path} ({size_mb:.1f} MB)")
        else:  # Directory
            file_count = len(list(path.glob('*.parquet')))
            print(f"✅ {file_path} ({file_count} parquet files)")
    else:
        print(f"❌ {file_path} (missing)")

# Check .gitkeep files in data subdirectories
print("\n📄 .gitkeep Files Status:")
print("=" * 40)

gitkeep_locations = [
    'data/interim/.gitkeep',
    'data/processed/.gitkeep',
    'db/.gitkeep',
    'outputs/.gitkeep'
]

for gitkeep_path in gitkeep_locations:
    path = Path(gitkeep_path)
    if path.exists():
        print(f"✅ {gitkeep_path}")
        # Show content
        content = path.read_text().strip()
        print(f"    Content: {content[:50]}...")
    else:
        print(f"❌ {gitkeep_path} (missing)")

# Show data directory tree structure
print("\n🌳 Data Directory Tree:")
print("=" * 40)
data_dir = Path('data')
if data_dir.exists():
    for root, dirs, files in os.walk(data_dir):
        level = root.replace(str(data_dir), '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}📁 {os.path.basename(root)}/")
        subindent = ' ' * 2 * (level + 1)
        for file in files[:5]:  # Show first 5 files
            print(f"{subindent}📄 {file}")
        if len(files) > 5:
            print(f"{subindent}... and {len(files) - 5} more files")

## 6. NBA Data Exploration

Now let's explore what's actually in our NBA datasets to understand the data structure and content.

In [ ]:
import pandas as pd
import numpy as np

# Load and analyze Players data
players_df = pd.read_parquet('data/raw/2024_NBA_Players.parquet')

print(f"Players Data Shape: {players_df.shape}")
print("Players Data Structure:")

# Create players column summary DataFrame
players_column_info = pd.DataFrame({
    'Column': players_df.columns,
    'Data_Type': [str(dtype) for dtype in players_df.dtypes.values],
    'Null_Count': [int(count) for count in players_df.isnull().sum().values],
    'Null_Percentage': [round(pct, 2) for pct in ((players_df.isnull().sum() / len(players_df)) * 100).values],
    'Unique_Count': [int(players_df[col].nunique()) for col in players_df.columns]
})

players_column_info

In [ ]:
def get_rows_with_nulls(df):
    """
    Returns a DataFrame containing all rows that have at least one null/missing value.
    
    Parameters:
    df (pandas.DataFrame): Input DataFrame to check for null values
    
    Returns:
    pandas.DataFrame: DataFrame containing only rows with null values
    """
    # Find rows where any column has null values
    null_mask = df.isnull().any(axis=1)
    
    # Return rows with null values
    rows_with_nulls = df[null_mask]
    
    print(f"Found {len(rows_with_nulls)} rows with null values out of {len(df)} total rows ({len(rows_with_nulls)/len(df)*100:.1f}%)")
    
    return rows_with_nulls

# Example usage:
# null_rows = get_rows_with_nulls(your_dataframe)

In [ ]:
players_with_null_df = get_rows_with_nulls(players_df)
players_with_null_df

In [ ]:
# Players data sample
print("Players Data Sample:")
players_df.head()

In [ ]:
# Load and analyze Schedule data
schedule_df = pd.read_parquet('data/raw/2024_NBA_Schedule.parquet')

print(f"Schedule Data Shape: {schedule_df.shape}")
print("Schedule Data Structure:")

# Create schedule column summary DataFrame with safe unique count calculation
def safe_unique_count(series):
    try:
        return int(series.nunique())
    except TypeError:
        # Handle unhashable types by converting to string first
        return int(series.astype(str).nunique())

columns = list(schedule_df.columns)
schedule_column_info = pd.DataFrame({
    'Column': columns,
    'Data_Type': [str(schedule_df[col].dtype) for col in columns],
    'Null_Count': [int(schedule_df[col].isnull().sum()) for col in columns],
    'Null_Percentage': [round(schedule_df[col].isnull().sum() / len(schedule_df) * 100, 2) for col in columns],
    'Unique_Count': [safe_unique_count(schedule_df[col]) for col in columns]
})

schedule_column_info

In [ ]:
# Schedule data sample
print("Schedule Data Sample:")
schedule_df.head()

In [ ]:
# Pacers games analysis
if 'HOME_TEAM_ABBREVIATION' in schedule_df.columns:
    pacers_home = schedule_df[schedule_df['HOME_TEAM_ABBREVIATION'] == 'IND']
    pacers_away = schedule_df[schedule_df['VISITOR_TEAM_ABBREVIATION'] == 'IND'] if 'VISITOR_TEAM_ABBREVIATION' in schedule_df.columns else pd.DataFrame()
    
    pacers_games_summary = pd.DataFrame({
        'Game_Type': ['Home Games', 'Away Games', 'Total Games'],
        'Count': [len(pacers_home), len(pacers_away), len(pacers_home) + len(pacers_away)]
    })
    
    print("Pacers Games Summary:")
    pacers_games_summary

In [ ]:
# Load and analyze Boxscores data
import glob

boxscore_files = glob.glob('data/raw/Boxscores/*.parquet')
print(f"Number of boxscore files: {len(boxscore_files)}")

if boxscore_files:
    # Read first boxscore file to understand structure
    first_file = boxscore_files[0]
    boxscore_df = pd.read_parquet(first_file)
    
    print(f"Sample file: {first_file.split('/')[-1]}")
    print(f"Boxscore Data Shape: {boxscore_df.shape}")
    print("Boxscore Data Structure:")
    
    # Create boxscore column summary DataFrame
    boxscore_column_info = pd.DataFrame({
        'Column': boxscore_df.columns,
        'Data_Type': [str(dtype) for dtype in boxscore_df.dtypes.values],
        'Null_Count': [int(count) for count in boxscore_df.isnull().sum().values],
        'Null_Percentage': [round(pct, 2) for pct in ((boxscore_df.isnull().sum() / len(boxscore_df)) * 100).values],
        'Unique_Count': [int(boxscore_df[col].nunique()) for col in boxscore_df.columns]
    })
    
    boxscore_column_info

In [ ]:
# Boxscore data sample
print("Boxscore Data Sample:")
boxscore_df.head()

In [ ]:
# Pacers players in boxscore
if 'TEAM_ABBREVIATION' in boxscore_df.columns:
    pacers_players = boxscore_df[boxscore_df['TEAM_ABBREVIATION'] == 'IND']
    
    if not pacers_players.empty:
        print(f"Pacers players in this game: {len(pacers_players)}")
        print("Sample Pacers player performance:")
        
        # Show key stats if available
        key_stats = ['PLAYER_NAME', 'MIN', 'PTS', 'REB', 'AST']
        available_stats = [col for col in key_stats if col in pacers_players.columns]
        
        if available_stats:
            pacers_players[available_stats].head()
        else:
            pacers_players.head()